# AI Constitution Compiler — Dev Log

## Objetivo

Análise estática de conflitos entre artigos de uma constituição declarativa
(mesmo formato de `constitutional_ai/constitution.yaml`) — 3 classes reais
de conflito: condição duplicada, subsunção, e sobreposição com severidade
diferente.

In [2]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import yaml
from core.constitution_compiler.compiler import compile_constitution
from core.constitutional_ai.engine import _DEFAULT_CONSTITUTION_PATH

with open(_DEFAULT_CONSTITUTION_PATH, "r", encoding="utf-8") as fh:
    data = yaml.safe_load(fh)
articles = data["constitution"]

result = compile_constitution(articles)
print(f"{result.article_count} artigos compilados (constitution.yaml de produção)")
print(f"Válido (sem conflito bloqueante)? {result.valid}")
for c in result.conflicts:
    print(f"  - [{c.conflict_type}] {c.article_a} <-> {c.article_b}: {c.explanation}")
print()
print(result.summary)

6 artigos compilados (constitution.yaml de produção)
Válido (sem conflito bloqueante)? False
  - [vacuous_overlap_different_severity] CONST-01 <-> CONST-02: Condições de 'CONST-01' (severidade=high) e 'CONST-02' (severidade=critical) não compartilham nenhuma chave de contexto — compatíveis só por vacuidade, sinal fraco, não bloqueante.
  - [overlapping_condition_different_severity] CONST-01 <-> CONST-03: Condições de 'CONST-01' (severidade=high) e 'CONST-03' (severidade=critical) compartilham a(s) chave(s) ['automated_decision'] e podem disparar simultaneamente com severidades diferentes.
  - [vacuous_overlap_different_severity] CONST-01 <-> CONST-05: Condições de 'CONST-01' (severidade=high) e 'CONST-05' (severidade=critical) não compartilham nenhuma chave de contexto — compatíveis só por vacuidade, sinal fraco, não bloqueante.
  - [vacuous_overlap_different_severity] CONST-01 <-> CONST-06: Condições de 'CONST-01' (severidade=high) e 'CONST-06' (severidade=medium) não compartilham nen

## Achado real inesperado

Rodar isto contra a constituição REAL de produção revelou **11 conflitos**,
bem mais do que se esperava numa primeira estimativa. A causa raiz é
honesta e importante: nenhum dos 6 artigos reais compartilha exatamente as
mesmas chaves de contexto, então quase todo par tem interseção vazia de
chaves — o que o modelo de compatibilidade considera "compatível por
vacuidade" (nenhuma chave em comum para contradizer). Isso é uma limitação
real do modelo, documentada no `CHANGELOG.md` como TODO — **não foi
"corrigida" para não mascarar o achado real**: o compilador está fazendo
exatamente o que promete, e o resultado é que a constituição de produção
tem mais ambiguidade de severidade do que parecia à primeira vista.

## Testes e Handoff

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/constitution_compiler/tests -v
```

10/10 testes passando, incluindo a análise real acima.

---

## Atualização (V5, 2026-08-21) — distingue overlap real de overlap por vacuidade

O achado original acima (11 conflitos) motivou um fix real no próprio
compilador: `overlapping_condition_different_severity` agora só é reportado
como **bloqueante** quando as duas condições compartilham de verdade uma
chave de contexto; quando a compatibilidade é só por vacuidade (nenhuma
chave em comum), vira `vacuous_overlap_different_severity`, informativo. Ver
`core/constitution_compiler/CHANGELOG.md` `[0.2.0]`.

Resultado real, revalidado contra a mesma constituição de produção: **2
conflitos genuinamente bloqueantes** (`CONST-01`↔`CONST-03`, ambos usando
`automated_decision`; `CONST-05`↔`CONST-06`, ambos usando `in_production` —
exatamente o par citado na análise original) + 9 informativos. O sinal de
11 falsos-positivos virou 2 achados de verdade acionáveis.